In [0]:
from pyspark.sql import functions as F

df = spark.read.table("hive_metastore.gold.gold_features")

df.select(
    F.count("*").alias("total_rows"),
    F.sum(F.when(F.col("load_ratio_c") > 1, 1).otherwise(0)).alias("current_overloads"),
    F.sum(F.when(F.col("load_ratio_v") > 1, 1).otherwise(0)).alias("voltage_overloads"),
    F.sum(F.when((F.col("load_ratio_c") > 1) | (F.col("load_ratio_v") > 1), 1).otherwise(0)).alias("any_overload"),
    F.sum(F.when((F.col("load_ratio_c") > 1) & (F.col("load_ratio_v") > 1), 1).otherwise(0)).alias("both_overload"),
).show()

In [0]:
from pyspark.sql import functions as F

df = spark.read.table("hive_metastore.gold.gold_dataset")

df.select(
    F.count("*").alias("total_rows"),
    F.sum(F.when(F.col("current") > F.col("H_LIM_C"), 1).otherwise(0)).alias("current_above_limit"),
    F.sum(F.when(F.col("voltage") > F.col("H_LIM_V"), 1).otherwise(0)).alias("voltage_above_limit"),
).show()

In [0]:
# =============================================================================
# Thesis Figure: Load Ratio Time Series for a Representative Transformer
# =============================================================================
# Run this in a Databricks notebook cell
# Produces two figures:
#   - Figure 3: Full year view (seasonal patterns + overload rarity)
#   - Figure 4: 2-week zoom around overload events (15-min granularity)
# =============================================================================

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
from pyspark.sql import functions as F

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
SRC_TABLE = "hive_metastore.gold.gold_dataset"  # Use gold_dataset (pre-scaling)
ID_COL = "ID_prefix"
TS_COL = "DATE"

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1: Find a "representative" transformer
# Criteria: median annual load, has some overload events, complete coverage
# ─────────────────────────────────────────────────────────────────────────────

df = spark.read.table(SRC_TABLE)

# Compute load_ratio if not already present
if "load_ratio_c" not in df.columns:
    df = df.withColumn(
        "load_ratio_c",
        F.when(F.col("H_LIM_C") > 0, F.col("current") / F.col("H_LIM_C")).otherwise(None)
    )

# Aggregate stats per transformer
transformer_stats = (
    df.groupBy(ID_COL)
    .agg(
        F.mean("load_ratio_c").alias("mean_load"),
        F.max("load_ratio_c").alias("max_load"),
        F.sum(F.when(F.col("load_ratio_c") > 1, 1).otherwise(0)).alias("n_overloads"),
        F.count("*").alias("n_rows"),
        F.min(TS_COL).alias("min_date"),
        F.max(TS_COL).alias("max_date"),
    )
    .withColumn("coverage_days", F.datediff(F.col("max_date"), F.col("min_date")))
    .filter(F.col("coverage_days") >= 360)  # At least ~1 year coverage
    .filter(F.col("n_overloads") > 10)       # Has some overloads to show
    .filter(F.col("n_overloads") < 5000)     # Not constantly overloaded
    .orderBy(F.abs(F.col("mean_load") - F.lit(0.5)))  # Closest to median load
)

print("Top 5 candidate transformers:")
transformer_stats.select(
    ID_COL, "mean_load", "max_load", "n_overloads", "n_rows", "coverage_days"
).show(5, truncate=False)

# Select the best candidate
selected_id = transformer_stats.first()[ID_COL]
print(f"\nSelected transformer: {selected_id}")



In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 2: Extract data for the selected transformer
# ─────────────────────────────────────────────────────────────────────────────

df_single = (
    df.filter(F.col(ID_COL) == selected_id)
    .select(TS_COL, "load_ratio_c", "current", "H_LIM_C")
    .orderBy(TS_COL)
    .toPandas()
)

df_single[TS_COL] = pd.to_datetime(df_single[TS_COL])
df_single = df_single.set_index(TS_COL).sort_index()

print(f"Data range: {df_single.index.min()} to {df_single.index.max()}")
print(f"Total rows: {len(df_single):,}")
print(f"Overload events (load_ratio_c > 1): {(df_single['load_ratio_c'] > 1).sum():,}")



In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 3: Find a good 2-week window with overload events
# ─────────────────────────────────────────────────────────────────────────────

# Find dates with overloads
overload_dates = df_single[df_single["load_ratio_c"] > 1].index

# Pick a 2-week window around an overload cluster (prefer summer months)
summer_overloads = overload_dates[(overload_dates.month >= 6) & (overload_dates.month <= 8)]
if len(summer_overloads) > 0:
    center_date = summer_overloads[len(summer_overloads) // 2]
else:
    center_date = overload_dates[len(overload_dates) // 2]

zoom_start = center_date - pd.Timedelta(days=7)
zoom_end = center_date + pd.Timedelta(days=7)

print(f"\nZoom window: {zoom_start.date()} to {zoom_end.date()}")






In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 4: Create Figure 3 — Full Year View
# ─────────────────────────────────────────────────────────────────────────────

fig1, ax1 = plt.subplots(figsize=(12, 4), dpi=150)

# Plot load ratio
ax1.plot(df_single.index, df_single["load_ratio_c"], 
         linewidth=0.3, color="#1f77b4", alpha=0.7, label="Load ratio (current)")

# Threshold line
ax1.axhline(y=1.0, color="#d62728", linestyle="--", linewidth=1.5, label="Overload threshold")

# Highlight overload points
overload_mask = df_single["load_ratio_c"] > 1
ax1.scatter(df_single.index[overload_mask], df_single.loc[overload_mask, "load_ratio_c"],
            color="#d62728", s=3, alpha=0.6, zorder=5, label="Overload events")

# Formatting
ax1.set_xlabel("Date", fontsize=11)
ax1.set_ylabel("Load Ratio (C / C_limit)", fontsize=11)
ax1.set_ylim(0, min(df_single["load_ratio_c"].max() * 1.1, 3.0))
ax1.xaxis.set_major_locator(mdates.MonthLocator())
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
ax1.legend(loc="upper right", fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.set_title(f"Transformer {selected_id} — Load Ratio Over 12 Months", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig("/dbfs/tmp/thesis_fig3_full_year.png", dpi=300, bbox_inches="tight")
plt.show()

print("\nSaved: /dbfs/tmp/thesis_fig3_full_year.png")


In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 5: Create Figure 4 — 2-Week Zoom
# ─────────────────────────────────────────────────────────────────────────────

df_zoom = df_single.loc[zoom_start:zoom_end]

fig2, ax2 = plt.subplots(figsize=(12, 4), dpi=150)

# Plot load ratio
ax2.plot(df_zoom.index, df_zoom["load_ratio_c"], 
         linewidth=0.8, color="#1f77b4", alpha=0.9, label="Load ratio (current)")

# Threshold line
ax2.axhline(y=1.0, color="#d62728", linestyle="--", linewidth=1.5, label="Overload threshold")

# Highlight overload points
overload_mask = df_zoom["load_ratio_c"] > 1
if overload_mask.sum() > 0:
    ax2.scatter(df_zoom.index[overload_mask], df_zoom.loc[overload_mask, "load_ratio_c"],
                color="#d62728", s=20, alpha=0.8, zorder=5, label="Overload events")

# Shade overload regions
for idx in df_zoom.index[overload_mask]:
    ax2.axvspan(idx, idx + pd.Timedelta(minutes=15), color="#d62728", alpha=0.1)

# Formatting
ax2.set_xlabel("Date", fontsize=11)
ax2.set_ylabel("Load Ratio (C / C_limit)", fontsize=11)
ax2.set_ylim(0, min(df_zoom["load_ratio_c"].max() * 1.1, 2.5))
ax2.xaxis.set_major_locator(mdates.DayLocator(interval=2))
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
ax2.xaxis.set_minor_locator(mdates.HourLocator(interval=12))
ax2.legend(loc="upper right", fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_title(f"Transformer {selected_id} — 2-Week Detail ({zoom_start.strftime('%d %b')} – {zoom_end.strftime('%d %b %Y')})", 
              fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig("/dbfs/tmp/thesis_fig4_zoom.png", dpi=300, bbox_inches="tight")
plt.show()

print("\nSaved: /dbfs/tmp/thesis_fig4_zoom.png")

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 6: Print summary stats for figure captions
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "="*60)
print("SUMMARY FOR FIGURE CAPTIONS")
print("="*60)
print(f"Transformer ID: {selected_id}")
print(f"Data period: {df_single.index.min().strftime('%d %B %Y')} – {df_single.index.max().strftime('%d %B %Y')}")
print(f"Total measurements: {len(df_single):,}")
print(f"Overload events (load_ratio > 1): {(df_single['load_ratio_c'] > 1).sum():,}")
print(f"Overload rate: {100 * (df_single['load_ratio_c'] > 1).mean():.2f}%")
print(f"Mean load ratio: {df_single['load_ratio_c'].mean():.3f}")
print(f"Max load ratio: {df_single['load_ratio_c'].max():.3f}")
print(f"Zoom window: {zoom_start.strftime('%d %B')} – {zoom_end.strftime('%d %B %Y')}")
print(f"Overloads in zoom window: {overload_mask.sum()}")

In [0]:
# =============================================================================
# Thesis Figure: Yearly Average Load Ratio Across All Transformers
# =============================================================================
# Run this in a Databricks notebook cell
# Produces:
#   - Figure: Daily average load_ratio_c and load_ratio_v across all transformers
#   - Shows seasonal patterns at the fleet level
# =============================================================================

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
from pyspark.sql import functions as F

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
SRC_TABLE = "hive_metastore.gold.gold_dataset"  # Use gold_dataset (pre-scaling)
TS_COL = "DATE"

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1: Load data and compute load ratios if needed
# ─────────────────────────────────────────────────────────────────────────────

df = spark.read.table(SRC_TABLE)
# Exclude rows where current or voltage is zero
df = df.filter((F.col("current") > 0) & (F.col("voltage") > 0))
# Compute load_ratio if not already present
if "load_ratio_c" not in df.columns:
    df = df.withColumn(
        "load_ratio_c",
        F.when(F.col("H_LIM_C") > 0, F.col("current") / F.col("H_LIM_C")).otherwise(None)
    )
if "load_ratio_v" not in df.columns:
    df = df.withColumn(
        "load_ratio_v",
        F.when(F.col("H_LIM_V") > 0, F.col("voltage") / F.col("H_LIM_V")).otherwise(None)
    )



In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 2: Aggregate by day — mean, max, and overload rate
# ─────────────────────────────────────────────────────────────────────────────

daily_stats = (
    df.withColumn("date", F.to_date(TS_COL))
    .groupBy("date")
    .agg(
        # Mean load ratios
        F.mean("load_ratio_c").alias("mean_load_ratio_c"),
        F.mean("load_ratio_v").alias("mean_load_ratio_v"),
        # Max load ratios (daily peak)
        F.max("load_ratio_c").alias("max_load_ratio_c"),
        F.max("load_ratio_v").alias("max_load_ratio_v"),
        # Overload rate (% of measurements exceeding threshold)
        F.mean(F.when(F.col("load_ratio_c") > 1, 1).otherwise(0)).alias("overload_rate_c"),
        F.mean(F.when(F.col("load_ratio_v") > 1, 1).otherwise(0)).alias("overload_rate_v"),
        # Count
        F.count("*").alias("n_measurements"),
    )
    .orderBy("date")
    .toPandas()
)

daily_stats["date"] = pd.to_datetime(daily_stats["date"])
daily_stats = daily_stats.set_index("date").sort_index()

print(f"Date range: {daily_stats.index.min().date()} to {daily_stats.index.max().date()}")
print(f"Total days: {len(daily_stats)}")

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 3: Figure — Mean Load Ratios (Current and Voltage)
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 1, figsize=(12, 7), dpi=150, sharex=True)

# --- Panel A: Load Ratio (Current) ---
ax1 = axes[0]
ax1.plot(daily_stats.index, daily_stats["mean_load_ratio_c"], 
         linewidth=0.8, color="#1f77b4", alpha=0.7, label="Daily mean")

# Add 7-day rolling average for trend
rolling_mean_c = daily_stats["mean_load_ratio_c"].rolling(window=7, center=True).mean()
ax1.plot(daily_stats.index, rolling_mean_c, 
         linewidth=2, color="#1f77b4", label="7-day rolling mean")

# Threshold line
ax1.axhline(y=1.0, color="#d62728", linestyle="--", linewidth=1.5, label="Overload threshold")

ax1.set_ylabel("Load Ratio (C / C_limit)", fontsize=11)
ax1.set_ylim(0, min(daily_stats["mean_load_ratio_c"].max() * 1.3, 1.2))
ax1.legend(loc="upper right", fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.set_title("(a) Current Load Ratio — Daily Average Across All Transformers", fontsize=11, fontweight="bold")

# --- Panel B: Load Ratio (Voltage) ---
ax2 = axes[1]
ax2.plot(daily_stats.index, daily_stats["mean_load_ratio_v"], 
         linewidth=0.8, color="#ff7f0e", alpha=0.7, label="Daily mean")

# Add 7-day rolling average for trend
rolling_mean_v = daily_stats["mean_load_ratio_v"].rolling(window=7, center=True).mean()
ax2.plot(daily_stats.index, rolling_mean_v, 
         linewidth=2, color="#ff7f0e", label="7-day rolling mean")

# Threshold line
ax2.axhline(y=1.0, color="#d62728", linestyle="--", linewidth=1.5, label="Overload threshold")

ax2.set_xlabel("Date", fontsize=11)
ax2.set_ylabel("Load Ratio (V / V_limit)", fontsize=11)
ax2.set_ylim(0, min(daily_stats["mean_load_ratio_v"].max() * 1.3, 1.2))
ax2.xaxis.set_major_locator(mdates.MonthLocator())
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
ax2.legend(loc="upper right", fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_title("(b) Voltage Load Ratio — Daily Average Across All Transformers", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.savefig("/dbfs/tmp/thesis_fig_avg_load_ratios.png", dpi=300, bbox_inches="tight")
plt.show()

print("\nSaved: /dbfs/tmp/thesis_fig_avg_load_ratios.png")

In [0]:
from pyspark.sql import functions as F

df = spark.read.table("hive_metastore.gold.gold_dataset")

# Check zeros
df.select(
    F.count("*").alias("total"),
    F.sum(F.when(F.col("current") == 0, 1).otherwise(0)).alias("current_zero"),
    F.sum(F.when(F.col("voltage") == 0, 1).otherwise(0)).alias("voltage_zero"),
    F.sum(F.when((F.col("current") == 0) | (F.col("voltage") == 0), 1).otherwise(0)).alias("either_zero"),
).show()

# Actual overload counts (not averages)
df.select(
    F.count("*").alias("total_rows"),
    F.sum(F.when(F.col("current") > F.col("H_LIM_C"), 1).otherwise(0)).alias("current_above_limit"),
    F.sum(F.when(F.col("voltage") > F.col("H_LIM_V"), 1).otherwise(0)).alias("voltage_above_limit"),
).show()

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 4: Figure — Daily Overload Rate
# ─────────────────────────────────────────────────────────────────────────────

fig2, ax3 = plt.subplots(figsize=(12, 4), dpi=150)

# Plot overload rates (as percentage)
ax3.bar(daily_stats.index, daily_stats["overload_rate_c"] * 100, 
        width=1, color="#1f77b4", alpha=0.6, label="Current overload rate")
ax3.bar(daily_stats.index, daily_stats["overload_rate_v"] * 100, 
        width=1, color="#ff7f0e", alpha=0.6, label="Voltage overload rate")

# Add 7-day rolling average
rolling_rate_c = (daily_stats["overload_rate_c"] * 100).rolling(window=7, center=True).mean()
ax3.plot(daily_stats.index, rolling_rate_c, 
         linewidth=2, color="#1f77b4", label="Current (7-day avg)")

rolling_rate_v = (daily_stats["overload_rate_v"] * 100).rolling(window=7, center=True).mean()
ax3.plot(daily_stats.index, rolling_rate_v, 
         linewidth=2, color="#ff7f0e", label="Voltage (7-day avg)")

ax3.set_xlabel("Date", fontsize=11)
ax3.set_ylabel("Overload Rate (%)", fontsize=11)
ax3.xaxis.set_major_locator(mdates.MonthLocator())
ax3.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
ax3.legend(loc="upper right", fontsize=9)
ax3.grid(True, alpha=0.3)
ax3.set_title("Daily Overload Rate Across All Transformers", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig("/dbfs/tmp/thesis_fig_overload_rate.png", dpi=300, bbox_inches="tight")
plt.show()

print("\nSaved: /dbfs/tmp/thesis_fig_overload_rate.png")

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 5: Print summary statistics
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "="*60)
print("SUMMARY STATISTICS FOR FIGURE CAPTIONS")
print("="*60)
print(f"\nDate range: {daily_stats.index.min().strftime('%d %B %Y')} – {daily_stats.index.max().strftime('%d %B %Y')}")
print(f"Total days: {len(daily_stats)}")

print(f"\n--- Current Load Ratio (load_ratio_c) ---")
print(f"Overall mean: {daily_stats['mean_load_ratio_c'].mean():.3f}")
print(f"Min daily mean: {daily_stats['mean_load_ratio_c'].min():.3f}")
print(f"Max daily mean: {daily_stats['mean_load_ratio_c'].max():.3f}")
print(f"Mean overload rate: {daily_stats['overload_rate_c'].mean() * 100:.3f}%")
print(f"Max daily overload rate: {daily_stats['overload_rate_c'].max() * 100:.2f}%")

print(f"\n--- Voltage Load Ratio (load_ratio_v) ---")
print(f"Overall mean: {daily_stats['mean_load_ratio_v'].mean():.3f}")
print(f"Min daily mean: {daily_stats['mean_load_ratio_v'].min():.3f}")
print(f"Max daily mean: {daily_stats['mean_load_ratio_v'].max():.3f}")
print(f"Mean overload rate: {daily_stats['overload_rate_v'].mean() * 100:.3f}%")
print(f"Max daily overload rate: {daily_stats['overload_rate_v'].max() * 100:.2f}%")

# Monthly breakdown
print("\n--- Monthly Overload Rates ---")
monthly = daily_stats.copy()
monthly["month"] = monthly.index.to_period("M")
monthly_stats = monthly.groupby("month").agg({
    "overload_rate_c": "mean",
    "overload_rate_v": "mean"
})
monthly_stats["overload_rate_c"] *= 100
monthly_stats["overload_rate_v"] *= 100
print(monthly_stats.round(3).to_string())

In [0]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
from pyspark.sql import functions as F
 
# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
# Try gold_features first (has events_15m_cnt), fall back to gold_dataset
SRC_TABLE = "hive_metastore.gold.gold_dataset"  # raw values, not scaled
TS_COL = "DATE"
EVENT_COL = "events_15m_cnt"  # The event count feature you engineered
 
# ─────────────────────────────────────────────────────────────────────────────
# STEP 1: Load data
# ─────────────────────────────────────────────────────────────────────────────
 
df = spark.read.table(SRC_TABLE)
 
# Check if event column exists
if EVENT_COL not in df.columns:
    print(f"Column {EVENT_COL} not found. Available columns:")
    print([c for c in df.columns if "event" in c.lower()])
    raise ValueError(f"Event column {EVENT_COL} not found")
 
print(f"Loaded table with {EVENT_COL} column")

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 2: Aggregate by day
# ─────────────────────────────────────────────────────────────────────────────
 
daily_events = (
    df.withColumn("date", F.to_date(TS_COL))
    .groupBy("date")
    .agg(
        F.sum(EVENT_COL).alias("total_events"),
        F.mean(EVENT_COL).alias("mean_events"),
        F.max(EVENT_COL).alias("max_events"),
        F.stddev(EVENT_COL).alias("std_events"),
        # Count rows with at least one event
        F.sum(F.when(F.col(EVENT_COL) > 0, 1).otherwise(0)).alias("rows_with_events"),
        F.count("*").alias("n_measurements"),
    )
    .withColumn("event_rate", F.col("rows_with_events") / F.col("n_measurements") * 100)
    .orderBy("date")
    .toPandas()
)
 
daily_events["date"] = pd.to_datetime(daily_events["date"])
daily_events = daily_events.set_index("date").sort_index()
 
print(f"Date range: {daily_events.index.min().date()} to {daily_events.index.max().date()}")
print(f"Total days: {len(daily_events)}")

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 3: Figure — Daily Total Events
# ─────────────────────────────────────────────────────────────────────────────
 
fig, axes = plt.subplots(2, 1, figsize=(12, 7), dpi=150, sharex=True)
 
# --- Panel A: Total daily events ---
ax1 = axes[0]
ax1.bar(daily_events.index, daily_events["total_events"], 
        width=1, color="#2ca02c", alpha=0.6, label="Daily total")
 
# Add 7-day rolling average
rolling_total = daily_events["total_events"].rolling(window=7, center=True).mean()
ax1.plot(daily_events.index, rolling_total, 
         linewidth=2, color="#2ca02c", label="7-day rolling mean")
 
ax1.set_ylabel("Total Event Count", fontsize=11)
ax1.legend(loc="upper right", fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.set_title("(a) Daily Total SCADA Events Across All Transformers", fontsize=11, fontweight="bold")
 
# --- Panel B: Event rate (% of measurements with events) ---
ax2 = axes[1]
ax2.bar(daily_events.index, daily_events["event_rate"], 
        width=1, color="#9467bd", alpha=0.6, label="Daily rate")
 
# Add 7-day rolling average
rolling_rate = daily_events["event_rate"].rolling(window=7, center=True).mean()
ax2.plot(daily_events.index, rolling_rate, 
         linewidth=2, color="#9467bd", label="7-day rolling mean")
 
ax2.set_xlabel("Date", fontsize=11)
ax2.set_ylabel("Event Rate (%)", fontsize=11)
ax2.xaxis.set_major_locator(mdates.MonthLocator())
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
ax2.legend(loc="upper right", fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_title("(b) Daily Rate of Measurements with Events (%)", fontsize=11, fontweight="bold")
 
plt.tight_layout()
plt.savefig("/dbfs/tmp/thesis_fig_event_counts.png", dpi=300, bbox_inches="tight")
plt.show()
 
print("\nSaved: /dbfs/tmp/thesis_fig_event_counts.png")

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 4: Correlation with overload rate (if you want to show relationship)
# ─────────────────────────────────────────────────────────────────────────────
 
# Load overload data for correlation
df_overload = spark.read.table(SRC_TABLE)
 
# Compute load ratios if needed (check if they exist and are unscaled)
# Note: gold_features may have scaled values, so we check
if "load_ratio_c" in df_overload.columns:
    daily_combined = (
        df_overload.withColumn("date", F.to_date(TS_COL))
        .groupBy("date")
        .agg(
            F.mean(EVENT_COL).alias("mean_events"),
            F.mean(F.when(F.col("load_ratio_c") > 1, 1).otherwise(0)).alias("overload_rate_c"),
            F.mean(F.when(F.col("load_ratio_v") > 1, 1).otherwise(0)).alias("overload_rate_v"),
        )
        .orderBy("date")
        .toPandas()
    )
    
    # Compute correlations
    corr_c = daily_combined["mean_events"].corr(daily_combined["overload_rate_c"])
    corr_v = daily_combined["mean_events"].corr(daily_combined["overload_rate_v"])
    
    print(f"\n--- Correlation: Events vs Overload Rate ---")
    print(f"Events vs Current overload rate: {corr_c:.3f}")
    print(f"Events vs Voltage overload rate: {corr_v:.3f}")

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 5: Print summary statistics
# ─────────────────────────────────────────────────────────────────────────────
 
print("\n" + "="*60)
print("SUMMARY STATISTICS FOR FIGURE CAPTIONS")
print("="*60)
print(f"\nDate range: {daily_events.index.min().strftime('%d %B %Y')} – {daily_events.index.max().strftime('%d %B %Y')}")
print(f"Total days: {len(daily_events)}")
 
print(f"\n--- Event Count Statistics ---")
print(f"Overall daily mean (total events): {daily_events['total_events'].mean():,.0f}")
print(f"Min daily total: {daily_events['total_events'].min():,.0f}")
print(f"Max daily total: {daily_events['total_events'].max():,.0f}")
print(f"Mean event rate: {daily_events['event_rate'].mean():.2f}%")
print(f"Max daily event rate: {daily_events['event_rate'].max():.2f}%")
 
# Monthly breakdown
print("\n--- Monthly Event Statistics ---")
monthly = daily_events.copy()
monthly["month"] = monthly.index.to_period("M")
monthly_stats = monthly.groupby("month").agg({
    "total_events": "mean",
    "event_rate": "mean"
})
monthly_stats.columns = ["mean_daily_events", "mean_event_rate"]
print(monthly_stats.round(2).to_string())

In [0]:
df = spark.read.table("hive_metastore.gold.gold_features")


# Quick correlation check: do events correlate with overloads?
df.select(
    F.corr("events_15m_cnt", "label_4h").alias("corr_4h"),
    F.corr("events_15m_cnt", "label_24h").alias("corr_24h"),
    F.corr("events_15m_cnt", "load_ratio_c").alias("corr_load_c"),
    F.corr("events_15m_cnt", "load_ratio_v").alias("corr_load_v")
).show()

In [0]:
# =============================================================================
# Thesis Figure: Weather Variables Throughout the Year
# =============================================================================
# Run this in a Databricks notebook cell
# Produces: 4-panel figure showing daily average weather variables
# =============================================================================

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
from pyspark.sql import functions as F

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
SRC_TABLE = "hive_metastore.gold.gold_dataset"  # or gold_features
TS_COL = "DATE"

# Weather column names (check your actual column names)
TEMP_COL = "temperatura_media_do_ar_horaria_c"
HUMIDITY_COL = "humidade_relativa_media_horaria_percent"
PRECIP_COL = "precipitacao_horaria_mm"
WIND_COL = "velocidade_do_vento_media_horaria_m_per_s"  # or m/s

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1: Load and aggregate by day
# ─────────────────────────────────────────────────────────────────────────────

df = spark.read.table(SRC_TABLE)

# Check column names
print("Available columns with 'temp', 'humid', 'precip', 'vento':")
for c in df.columns:
    if any(x in c.lower() for x in ['temp', 'humid', 'precip', 'vento', 'wind']):
        print(f"  {c}")

# Aggregate by day
daily_weather = (
    df.withColumn("date", F.to_date(TS_COL))
    .withColumn("hour", F.hour(TS_COL))
    .dropDuplicates(["ID_prefix", "date", "hour"])  # one row per transformer-hour
    .groupBy("date")
    .agg(
        F.mean(TEMP_COL).alias("temperature"),
        F.mean(HUMIDITY_COL).alias("humidity"),
        F.mean(PRECIP_COL).alias("precipitation"),  # now sum is correct
        F.mean(WIND_COL).alias("wind_speed"),
    )
    .orderBy("date")
    .toPandas()
)

daily_weather["date"] = pd.to_datetime(daily_weather["date"])
daily_weather = daily_weather.set_index("date").sort_index()

print(f"Date range: {daily_weather.index.min().date()} to {daily_weather.index.max().date()}")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2: Create 4-panel figure
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(4, 1, figsize=(12, 10), dpi=150, sharex=True)

# Color scheme
colors = ["#e41a1c", "#377eb8", "#4daf4a", "#984ea3"]  # red, blue, green, purple

# --- Panel A: Temperature ---
ax1 = axes[0]
ax1.plot(daily_weather.index, daily_weather["temperature"], 
         linewidth=0.8, color=colors[0], alpha=0.6)
rolling_temp = daily_weather["temperature"].rolling(window=7, center=True).mean()
ax1.plot(daily_weather.index, rolling_temp, linewidth=2, color=colors[0], label="7-day avg")
ax1.set_ylabel("Temperature (°C)", fontsize=10)
ax1.legend(loc="upper right", fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.set_title("(a) Mean Daily Air Temperature", fontsize=11, fontweight="bold")

# --- Panel B: Humidity ---
ax2 = axes[1]
ax2.plot(daily_weather.index, daily_weather["humidity"], 
         linewidth=0.8, color=colors[1], alpha=0.6)
rolling_humid = daily_weather["humidity"].rolling(window=7, center=True).mean()
ax2.plot(daily_weather.index, rolling_humid, linewidth=2, color=colors[1], label="7-day avg")
ax2.set_ylabel("Humidity (%)", fontsize=10)
ax2.legend(loc="upper right", fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_title("(b) Mean Daily Relative Humidity", fontsize=11, fontweight="bold")

# --- Panel C: Precipitation ---
ax3 = axes[2]
ax3.bar(daily_weather.index, daily_weather["precipitation"], 
        width=1, color=colors[2], alpha=0.7)
rolling_precip = daily_weather["precipitation"].rolling(window=7, center=True).mean()
ax3.plot(daily_weather.index, rolling_precip, linewidth=2, color=colors[2], label="7-day avg")
ax3.set_ylabel("Precipitation (mm)", fontsize=10)
ax3.legend(loc="upper right", fontsize=9)
ax3.grid(True, alpha=0.3)
ax3.set_title("(c) Mean Daily Precipitation", fontsize=11, fontweight="bold")

# --- Panel D: Wind Speed ---
ax4 = axes[3]
ax4.plot(daily_weather.index, daily_weather["wind_speed"], 
         linewidth=0.8, color=colors[3], alpha=0.6)
rolling_wind = daily_weather["wind_speed"].rolling(window=7, center=True).mean()
ax4.plot(daily_weather.index, rolling_wind, linewidth=2, color=colors[3], label="7-day avg")
ax4.set_xlabel("Date", fontsize=11)
ax4.set_ylabel("Wind Speed (m/s)", fontsize=10)
ax4.xaxis.set_major_locator(mdates.MonthLocator())
ax4.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
ax4.legend(loc="upper right", fontsize=9)
ax4.grid(True, alpha=0.3)
ax4.set_title("(d) Mean Daily Wind Speed", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.savefig("/dbfs/tmp/thesis_fig_weather.png", dpi=300, bbox_inches="tight")
plt.show()

print("\nSaved: /dbfs/tmp/thesis_fig_weather.png")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3: Summary statistics
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "="*60)
print("SUMMARY STATISTICS FOR FIGURE CAPTION")
print("="*60)
print(f"\nTemperature: {daily_weather['temperature'].mean():.1f}°C mean, "
      f"{daily_weather['temperature'].min():.1f}°C min, "
      f"{daily_weather['temperature'].max():.1f}°C max")
print(f"Humidity: {daily_weather['humidity'].mean():.1f}% mean, "
      f"{daily_weather['humidity'].min():.1f}% min, "
      f"{daily_weather['humidity'].max():.1f}% max")
print(f"Precipitation: {daily_weather['precipitation'].sum():.0f} mm total, "
      f"{daily_weather['precipitation'].max():.1f} mm max daily")
print(f"Wind speed: {daily_weather['wind_speed'].mean():.1f} m/s mean, "
      f"{daily_weather['wind_speed'].max():.1f} m/s max")

# Monthly breakdown
print("\n--- Monthly Averages ---")
monthly = daily_weather.copy()
monthly["month"] = monthly.index.to_period("M")
monthly_avg = monthly.groupby("month").mean()
print(monthly_avg.round(1).to_string())